<a href="https://colab.research.google.com/github/N-Uma/Machine-Learning-and-Big-Data/blob/main/big_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

path = "/content/drive/MyDrive/NYC_Taxi_Yellow_2012"
print(f"PROJECT: {path}")

import os
folders = [
    "notebooks", "data/raw", "data/processed",
    "tableau", "config", "scripts", "tests"
]

for folder in folders:
    folder_path = os.path.join(path, folder)
    os.makedirs(folder_path, exist_ok=True)
    print(f"Created: {folder_path}")

print("\n FOLDER STRUCTURE:")
os.listdir(path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT: /content/drive/MyDrive/NYC_Taxi_Yellow_2012
Created: /content/drive/MyDrive/NYC_Taxi_Yellow_2012/notebooks
Created: /content/drive/MyDrive/NYC_Taxi_Yellow_2012/data/raw
Created: /content/drive/MyDrive/NYC_Taxi_Yellow_2012/data/processed
Created: /content/drive/MyDrive/NYC_Taxi_Yellow_2012/tableau
Created: /content/drive/MyDrive/NYC_Taxi_Yellow_2012/config
Created: /content/drive/MyDrive/NYC_Taxi_Yellow_2012/scripts
Created: /content/drive/MyDrive/NYC_Taxi_Yellow_2012/tests

 FOLDER STRUCTURE:


['notebooks',
 'data',
 'tableau',
 'config',
 'scripts',
 'tests',
 'models',
 'final_dataset.csv']

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

path = "/content/drive/MyDrive/NYC_Taxi_Yellow_2012"


if 'spark' in locals() and spark:
    try:
        spark.stop()
        print("Existing Spark session stopped successfully.")
    except Exception as e:
        print(f"Could not stop existing Spark session (it might already be stopped or unreachable): {e}")

spark = (
    SparkSession.builder
    .appName("taxi-ingestion")
    .getOrCreate()
)

sc = spark.sparkContext
print(f"Version: {spark.version}")
print(f"Spark UI: {sc.uiWebUrl}")

Version: 4.0.2
Spark UI: http://eba9bc008242:4040


In [ ]:
raw_data = f"{path}/data/raw/"

import os
files = os.listdir(raw_data)
parquet_files = [file for file in files if file.endswith('.parquet')]
print(f"Found {len(parquet_files)} parquet files:")
for file in parquet_files[:]:
    print(f" {file}")

Found 12 parquet files:
 yellow_2012_01.parquet
 yellow_2012_02.parquet
 yellow_2012_03.parquet
 yellow_2012_04.parquet
 yellow_2012_05.parquet
 yellow_2012_06.parquet
 yellow_2012_07.parquet
 yellow_2012_08.parquet
 yellow_2012_09.parquet
 yellow_2012_10.parquet
 yellow_2012_11.parquet
 yellow_2012_12.parquet


In [ ]:
yellowtaxi_df = spark.read.parquet(f"{path}/data/raw/*.parquet")
print("\n Sample data:")
yellowtaxi_df.show(7)
print("\n Schema:")
yellowtaxi_df.printSchema()



 Sample data:
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2012-01-01 00:07:56|  2012-01-01 00:12:09|              1|          0.9|         1|                 N|         158|         231|           2|        4.9|  0.5|  

In [ ]:
from pyspark.sql.types import DoubleType, FloatType

def null_check(df):
    rows_count = df.count()
    print(f"Total rows: {rows_count:,}")

    key_cols = ["fare_amount", "trip_distance", "passenger_count",
                "tpep_pickup_datetime"]

    select_expressions = []
    for c in key_cols:
        col_type = df.schema[c].dataType
        if isinstance(col_type, (DoubleType, FloatType)):
            select_expressions.append(
                sum(when(col(c).isNull() | isnan(col(c)), 1).otherwise(0)).alias(f"{c}_nulls")
            )
        else:
            select_expressions.append(
                sum(when(col(c).isNull(), 1).otherwise(0)).alias(f"{c}_nulls")
            )

    null_summary = df.select(select_expressions).collect()[0].asDict()

    print("\n NULL PERCENTAGE:")
    for col_name, null_count in null_summary.items():
        pct = (null_count / rows_count) * 100
        print(f"  {col_name:20s}: {null_count:8,} ({pct:6.2f}%)")

    print("\n FARE AMOUNT STATS:")
    df.select("fare_amount").describe().show()

    return rows_count, null_summary

rows_count, null_summary = null_check(yellowtaxi_df)

Total rows: 171,359,007

 NULL PERCENTAGE:
  fare_amount_nulls   :        0 (  0.00%)
  trip_distance_nulls :        0 (  0.00%)
  passenger_count_nulls:        0 (  0.00%)
  tpep_pickup_datetime_nulls:        0 (  0.00%)

 FARE AMOUNT STATS:
+-------+------------------+
|summary|       fare_amount|
+-------+------------------+
|  count|         171359007|
|   mean|10.989770782162557|
| stddev| 8.923958348729043|
|    min|               2.5|
|    max|             500.0|
+-------+------------------+



In [ ]:
clean_df = yellowtaxi_df.filter(
    (col("fare_amount").between(1.0, 500.0)) &
    (col("trip_distance").between(0.1, 100.0)) &
    (col("passenger_count").cast("int").between(1, 6))
).dropDuplicates()

In [ ]:
clean_df = clean_df.withColumn("pickup_month", date_format(col("tpep_pickup_datetime"), "yyyy-MM"))

In [ ]:
import os


RAW_PATH = raw_data


if os.path.isdir(RAW_PATH):
    print(f"Contents of '{RAW_PATH}':")
    for item in os.listdir(RAW_PATH):
        print(item)
else:
    print(f"The directory '{RAW_PATH}' does not exist.")

Contents of '/content/drive/MyDrive/NYC_Taxi_Yellow_2012/data/raw/':
yellow_2012_01.parquet
yellow_2012_02.parquet
yellow_2012_03.parquet
yellow_2012_04.parquet
yellow_2012_05.parquet
yellow_2012_06.parquet
yellow_2012_07.parquet
yellow_2012_08.parquet
yellow_2012_09.parquet
yellow_2012_10.parquet
yellow_2012_11.parquet
yellow_2012_12.parquet


In [ ]:
from pyspark.sql.functions import col, date_format, year
from pyspark import StorageLevel


columns_needed = [
    "fare_amount",
    "trip_distance",
    "passenger_count",
    "tpep_pickup_datetime",
    "congestion_surcharge"
]

df_pruned = yellowtaxi_df.select(*columns_needed)


df_clean = (
    df_pruned
    .filter(col("fare_amount").between(1.0, 500.0))
    .filter(col("trip_distance").between(0.1, 100.0))
    .filter(col("passenger_count").cast("int").between(1, 6))
    .filter(year(col("tpep_pickup_datetime").cast("timestamp")) == 2012)
)


df_clean = df_clean.withColumn(
    "pickup_month",
    date_format(col("tpep_pickup_datetime"), "yyyy-MM")
)


df_clean = df_clean.persist(StorageLevel.MEMORY_AND_DISK)


print("Cleaned data ready.")

Cleaned data ready.


In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, date_format, year
from pyspark.sql.types import *
from pyspark import StorageLevel


for df_name in ['df_clean', 'df_pruned', 'yellowtaxi_df']:
    if df_name in locals() and isinstance(locals()[df_name], pyspark.sql.DataFrame):
        try:
            locals()[df_name].unpersist()
            print(f"{df_name} unpersisted.")
        except:
            pass


spark.conf.set("spark.sql.parquet.enableVectorizedReader", "false")
print("spark.sql.parquet.enableVectorizedReader set to false.")


custom_schema = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("airport_fee", DoubleType(), True)
])


yellowtaxi_df = spark.read.schema(custom_schema).parquet(f"{path}/data/raw/*.parquet")
print("yellowtaxi_df re-read successfully with custom schema.")


columns_needed = [
    "fare_amount",
    "trip_distance",
    "passenger_count",
    "tpep_pickup_datetime",
    "congestion_surcharge"
]

df_pruned = yellowtaxi_df.select(*columns_needed)



df_clean = (
    df_pruned
    .filter(col("fare_amount").between(1.0, 500.0))
    .filter(col("trip_distance").between(0.1, 100.0))
    .filter(col("passenger_count").cast("int").between(1, 6))
    .filter(year(col("tpep_pickup_datetime").cast("timestamp")) == 2012)
)


print("df_clean DataFrame created (lazily).")

print("\nSchema of df_clean:")
df_clean.printSchema()

print("\nSample data from df_clean:")
df_clean.show(7)

df_clean unpersisted.
df_pruned unpersisted.
yellowtaxi_df unpersisted.
spark.sql.parquet.enableVectorizedReader set to false.
yellowtaxi_df re-read successfully with custom schema.
df_clean DataFrame created (lazily).

Schema of df_clean:
root
 |-- fare_amount: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- congestion_surcharge: double (nullable = true)


Sample data from df_clean:
+-----------+-------------+---------------+--------------------+--------------------+
|fare_amount|trip_distance|passenger_count|tpep_pickup_datetime|congestion_surcharge|
+-----------+-------------+---------------+--------------------+--------------------+
|        4.9|          0.9|              1| 2012-01-01 00:07:56|                NULL|
|        8.5|          2.3|              1| 2012-01-01 00:18:49|                NULL|
|        9.3|          2.2|              1| 2012-01-01 0

In [ ]:
from pyspark.sql.functions import col, date_format


PROCESSED_PATH = f"{path}/data/processed/"


spark.conf.set("spark.sql.parquet.enableVectorizedReader", "false")
print("spark.sql.parquet.enableVectorizedReader set to false before writing.")


df_clean_with_month = df_clean.withColumn(
    "pickup_month",
    date_format(col("tpep_pickup_datetime"), "yyyy-MM")
)


(
    df_clean_with_month
    .repartition(50)
    .write
    .mode("overwrite")
    .option("compression", "snappy")
    .partitionBy("pickup_month")
    .parquet(PROCESSED_PATH)
)

print(f" Data successfully written to: {PROCESSED_PATH}")

spark.sql.parquet.enableVectorizedReader set to false before writing.
 Data successfully written to: /content/drive/MyDrive/NYC_Taxi_Yellow_2012/data/processed/


In [ ]:

df_final = spark.read.parquet(PROCESSED_PATH)

print("--- Partition Distribution (Expected: only 2012 data) ---")

(
    df_final
    .groupBy("pickup_month")
    .count()
    .orderBy("pickup_month")
    .show(12, truncate=False)
)

--- Partition Distribution (Expected: only 2012 data) ---
+------------+--------+
|pickup_month|count   |
+------------+--------+
|2012-01     |12689698|
|2012-02     |12979847|
|2012-03     |15685434|
|2012-04     |12995743|
|2012-05     |13846759|
|2012-06     |14966148|
|2012-07     |14257696|
|2012-08     |14257648|
|2012-09     |14430991|
|2012-10     |14407354|
|2012-11     |13663801|
|2012-12     |14572953|
+------------+--------+



Feature Engineering

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml import Pipeline, Transformer
from pyspark.ml.feature import (
    VectorAssembler,
    StandardScaler,
    OneHotEncoder,
    StringIndexer
)
from pyspark.ml.util import DefaultParamsReadable, DefaultParamsWritable


BASE_PATH = path
INPUT_PATH = PROCESSED_PATH
OUTPUT_PATH = f"{BASE_PATH}/data/final"
MODEL_PATH = f"{BASE_PATH}/models/feature_pipeline_2012"


spark = (
    SparkSession.builder
    .appName("NYCTaxi_FeatureEngineering_2012")
    .config("spark.sql.parquet.enableVectorizedReader", "true")
    .getOrCreate()
)


df = spark.read.parquet(INPUT_PATH)



df_features = (
    df
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime"))
    .withColumn(
        "is_peak_hour",
        F.when(
            F.col("pickup_hour").between(7, 9) |
            F.col("pickup_hour").between(16, 19),
            1
        ).otherwise(0)
    )
    .withColumn(
        "is_weekend",
        F.when(F.col("day_of_week").isin(1, 7), 1).otherwise(0)
    )

    .fillna(0)
)



dow_indexer = StringIndexer(
    inputCol="day_of_week",
    outputCol="dow_index",
    handleInvalid="keep"
)

dow_encoder = OneHotEncoder(
    inputCol="dow_index",
    outputCol="dow_vector"
)

numeric_features = [
    "passenger_count",
    "trip_distance",
    "congestion_surcharge",
    "pickup_hour",
    "is_peak_hour",
    "is_weekend"
]

assembler = VectorAssembler(
    inputCols=numeric_features + ["dow_vector"],
    outputCol="features_unscaled"
)

scaler = StandardScaler(
    inputCol="features_unscaled",
    outputCol="features",
    withStd=True,
    withMean=True
)

pipeline = Pipeline(
    stages=[
        dow_indexer,
        dow_encoder,
        assembler,
        scaler
    ]
)


pipeline_model = pipeline.fit(df_features)
df_final_features = pipeline_model.transform(df_features)


train_df = df_final_features.filter(F.col("pickup_month") <= "2012-10")
val_df   = df_final_features.filter(F.col("pickup_month") == "2012-11")
test_df  = df_final_features.filter(F.col("pickup_month") == "2012-12")


def save_dataset(df, name):
    path = f"{OUTPUT_PATH}/{name}"
    (
        df
        .select("features", "fare_amount")
        .write
        .mode("overwrite")
        .parquet(path)
    )
    print(f" Saved {name} to {path}")

save_dataset(train_df, "train_2012")
save_dataset(val_df, "val_2012")
save_dataset(test_df, "test_2012")


pipeline_model.write().overwrite().save(MODEL_PATH)

print("\n--- Feature Engineering Complete ---")
df_final_features.select("features", "fare_amount").show(5, truncate=False)


 Saved train_2012 to /content/drive/MyDrive/NYC_Taxi_Yellow_2012/data/final/train_2012
 Saved val_2012 to /content/drive/MyDrive/NYC_Taxi_Yellow_2012/data/final/val_2012
 Saved test_2012 to /content/drive/MyDrive/NYC_Taxi_Yellow_2012/data/final/test_2012

--- Feature Engineering Complete ---
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|features                                                                                                                                                                                                                                                                 |fare_amount|
+--------------------------------------------------------------------------------------------------------------------------------------------------

Model Evaluation

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.regression import GBTRegressionModel
from pyspark.sql.types import *
import numpy as np


BASE_PATH = path


if "spark" in locals():
    try:
        spark.stop()
    except Exception as err:
        print(f"Warning: Spark session could not be stopped: {err}")


spark = (
    SparkSession.builder
    .appName("NYC Taxi Prediction")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .getOrCreate()
)

train_df = spark.read.parquet(f"{BASE_PATH}/data/final/train_2012")
val_df   = spark.read.parquet(f"{BASE_PATH}/data/final/val_2012")
test_df  = spark.read.parquet(f"{BASE_PATH}/data/final/test_2012")


train_count = train_df.count()
val_count   = val_df.count()
test_count  = test_df.count()
total_count = train_count + val_count + test_count

print(f"Train (Jan–Oct): {train_count:,} rows ({train_count / total_count * 100:.1f}")
print(f"Val (Nov):       {val_count:,} rows")
print(f"Test (Dec):      {test_count:,} rows")

Train (Jan–Oct): 140,517,318 rows (83.3
Val (Nov):       13,663,801 rows
Test (Dec):      14,572,953 rows


In [ ]:
from pyspark.ml.regression import RandomForestRegressor


rf_model = RandomForestRegressor(
    featuresCol="features",
    labelCol="fare_amount",
    numTrees=3,
    maxDepth=3,
    subsamplingRate=0.7,
    seed=42
)


best_model = rf_model.fit(train_df)

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql import Row

print("\nDISTRIBUTED REGRESSION METRICS\n")


test_preds = best_model.transform(test_df).cache()
test_preds.count()


evaluators = {
    "RMSE": RegressionEvaluator(
        labelCol="fare_amount",
        predictionCol="prediction",
        metricName="rmse"
    ),
    "MSE": RegressionEvaluator(
        labelCol="fare_amount",
        predictionCol="prediction",
        metricName="mse"
    ),
    "MAE": RegressionEvaluator(
        labelCol="fare_amount",
        predictionCol="prediction",
        metricName="mae"
    ),
    "R2": RegressionEvaluator(
        labelCol="fare_amount",
        predictionCol="prediction",
        metricName="r2"
    )
}


metrics = {}

for metric_name, evaluator in evaluators.items():
    value = evaluator.evaluate(test_preds)
    metrics[metric_name] = __builtins__.round(value, 4)
    print(f"{metric_name:<6}: {metrics[metric_name]}")


summary_df = spark.createDataFrame([Row(**metrics)])
print("\nSUMMARY TABLE:")
summary_df.show(truncate=False)


test_preds.unpersist()


DISTRIBUTED REGRESSION METRICS

RMSE  : 6.8395
MSE   : 46.7788
MAE   : 4.3883
R2    : 0.4974

SUMMARY TABLE:
+------+-------+------+------+
|RMSE  |MSE    |MAE   |R2    |
+------+-------+------+------+
|6.8395|46.7788|4.3883|0.4974|
+------+-------+------+------+



DataFrame[features: vector, fare_amount: double, prediction: double]

In [ ]:
from pyspark.sql.functions import col, avg, lit

print("\nTEMPORAL SPLIT ANALYSIS")
print("Temporal ordering confirmed:")
print("  Train: Jan–Oct")
print("  Val:   Nov")
print("  Test:  Dec")


monthly_perf = (
    test_preds
    .withColumn("pickup_month", lit("2015-12"))
    .groupBy("pickup_month")
    .agg(
        avg(col("prediction") - col("fare_amount")).alias("mean_error"),
        avg(col("fare_amount")).alias("avg_fare")
    )
)


monthly_perf.show(truncate=False)


TEMPORAL SPLIT ANALYSIS
Temporal ordering confirmed:
  Train: Jan–Oct
  Val:   Nov
  Test:  Dec
+------------+-------------------+------------------+
|pickup_month|mean_error         |avg_fare          |
+------------+-------------------+------------------+
|2015-12     |-1.5506257501653065|12.231407352373934|
+------------+-------------------+------------------+



In [ ]:
from pyspark.sql.functions import col, when
from pyspark.ml.evaluation import RegressionEvaluator
import numpy as np

print("\n5-FOLD CROSS-VALIDATION (APPROX. STRATIFIED)")


fare_buckets = (
    test_preds
    .withColumn(
        "fare_bucket",
        when(col("fare_amount") < 10, "low")
        .when(col("fare_amount") < 25, "medium")
        .otherwise("high")
    )
)


print("Fare distribution:")
(
    fare_buckets
    .groupBy("fare_bucket")
    .count()
    .orderBy("fare_bucket")
    .show()
)


evaluator = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="rmse"
)

cv_scores = []


for fold_id in range(5):

    fold_df = (
        fare_buckets
        .sample(
            withReplacement=False,
            fraction=0.2,
            seed=fold_id
        )
        .select("features", "fare_amount")
        .cache()
    )


    fold_df.count()

    fold_predictions = best_model.transform(fold_df)

    rmse = evaluator.evaluate(fold_predictions)
    cv_scores.append(rmse)

    fold_df.unpersist()


mean_rmse = np.mean(cv_scores)
std_rmse = np.std(cv_scores)

print(f"Mean RMSE: ${mean_rmse:.2f} ± {std_rmse:.2f}")
print("Fold RMSEs:", [f"{score:.2f}" for score in cv_scores])


5-FOLD CROSS-VALIDATION (APPROX. STRATIFIED)
Fare distribution:
+-----------+-------+
|fare_bucket|  count|
+-----------+-------+
|       high|1191475|
|        low|7767230|
|     medium|5614248|
+-----------+-------+

Mean RMSE: $6.84 ± 0.01
Fold RMSEs: ['6.85', '6.84', '6.82', '6.86', '6.84']


In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator
import numpy as np

print("\nBOOTSTRAP CONFIDENCE INTERVALS")


test_preds_cached = (
    best_model
    .transform(test_df.coalesce(10))
    .select("prediction", "fare_amount")
    .cache()
)


test_preds_cached.count()


evaluator = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="rmse"
)


def bootstrap_rmse_ci(n_bootstrap=50):
    rmse_scores = []

    for seed in range(n_bootstrap):
        bootstrap_sample = (
            test_preds_cached
            .sample(
                withReplacement=True,
                fraction=0.8,
                seed=seed
            )
        )

        rmse = evaluator.evaluate(bootstrap_sample)
        rmse_scores.append(rmse)

    rmse_scores = np.array(sorted(rmse_scores))

    lower_idx = int(0.025 * len(rmse_scores))
    upper_idx = int(0.975 * len(rmse_scores))

    mean_rmse = rmse_scores.mean()
    ci_lower = rmse_scores[lower_idx]
    ci_upper = rmse_scores[upper_idx]

    return mean_rmse, ci_lower, ci_upper


mean_rmse, ci_low, ci_high = bootstrap_rmse_ci(n_bootstrap=50)

print("RMSE Bootstrap Results:")
print(f"  Mean RMSE: ${mean_rmse:.2f}")
print(f"  95% CI:    ${ci_low:.2f} – ${ci_high:.2f}")
print(f"  CI Width:  ${ci_high - ci_low:.2f} (narrow = stable model)")


test_preds_cached.unpersist()


BOOTSTRAP CONFIDENCE INTERVALS
RMSE Bootstrap Results:
  Mean RMSE: $6.84
  95% CI:    $6.83 – $6.86
  CI Width:  $0.03 (narrow = stable model)


DataFrame[prediction: double, fare_amount: double]

In [ ]:
from pyspark.sql.functions import col, avg, abs as sql_abs, count, when

print("\nBUSINESS METRICS ALIGNMENT")


test_with_profit = (
    test_preds
    .select("fare_amount", "prediction")
    .withColumn("driver_profit", col("fare_amount"))
    .withColumn("predicted_profit", col("prediction"))
)

business_metrics_row = (
    test_with_profit
    .agg(
        avg(col("driver_profit")).alias("avg_driver_profit"),
        avg(col("predicted_profit")).alias("avg_pred_profit"),
        avg(sql_abs(col("driver_profit") - col("predicted_profit")))
            .alias("profit_forecast_error"),
        count(when(col("predicted_profit") > 0, True))
            .alias("profitable_trips_pred"),
        count(when(col("driver_profit") > 0, True))
            .alias("profitable_trips_actual")
    )
    .first()
)


print("NYC Taxi Driver Economics:")
print(f"Actual avg profit / trip:     ${business_metrics_row['avg_driver_profit']:.2f}")
print(f"Predicted avg profit / trip:  ${business_metrics_row['avg_pred_profit']:.2f}")
print(f"Profit forecast error:        ${business_metrics_row['profit_forecast_error']:.2f}")
print(f"Profitable trips (actual):    {business_metrics_row['profitable_trips_actual']:,}")
print(f"Profitable trips (predicted): {business_metrics_row['profitable_trips_pred']:,}")


BUSINESS METRICS ALIGNMENT
NYC Taxi Driver Economics:
Actual avg profit / trip:     $12.23
Predicted avg profit / trip:  $10.68
Profit forecast error:        $4.39
Profitable trips (actual):    14,572,953
Profitable trips (predicted): 14,572,953


Model Training

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.regression import (
    LinearRegression,
    GBTRegressor,
    RandomForestRegressor
)
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
import pyspark.sql.functions as F



BASE_PATH = "/content/drive/MyDrive/NYC_Taxi_Yellow_2012"


spark = SparkSession.builder.getOrCreate()
print(f"Spark {spark.version} ready")

train_df = spark.read.parquet(f"{BASE_PATH}/data/final/train_2012")
val_df   = spark.read.parquet(f"{BASE_PATH}/data/final/val_2012")
test_df  = spark.read.parquet(f"{BASE_PATH}/data/final/test_2012")

train_count = train_df.count()
val_count   = val_df.count()
test_count  = test_df.count()

print(f"Train: {train_count:,} rows | Schema: {train_df.columns}")
print(f"Val:   {val_count:,} rows")
print(f"Test:  {test_count:,} rows")

train_df.show(3, truncate=False)

Spark 4.0.2 ready
Train: 140,517,318 rows | Schema: ['features', 'fare_amount']
Val:   13,663,801 rows
Test:  14,572,953 rows
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|features                                                                                                                                                                                                                                                               |fare_amount|
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|[0.23388725290759546,0.9758800005497195

In [ ]:
from pyspark.ml.regression import (
    LinearRegression,
    GBTRegressor,
    RandomForestRegressor
)

models_config = {
    "1. Linear Regression": LinearRegression(
        featuresCol="features",
        labelCol="fare_amount",
        maxIter=50,
        regParam=0.1,
        elasticNetParam=0.5
    ),

    "2. Gradient Boosted Trees": GBTRegressor(
        featuresCol="features",
        labelCol="fare_amount",
        maxIter=20,
        maxDepth=6,
        stepSize=0.05
    ),

    "3. Random Forest": RandomForestRegressor(
        featuresCol="features",
        labelCol="fare_amount",
        numTrees=20,
        maxDepth=8,
        featureSubsetStrategy="sqrt"
    ),

    "4. Ridge Regression": LinearRegression(
        featuresCol="features",
        labelCol="fare_amount",
        maxIter=50,
        regParam=1.0,
        elasticNetParam=0.0
    )
}


print("Configured MLlib Regression Models:")
for model_name in models_config:
    print(f"  • {model_name}")

Configured MLlib Regression Models:
  • 1. Linear Regression
  • 2. Gradient Boosted Trees
  • 3. Random Forest
  • 4. Ridge Regression


In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator


train_small = (
    train_df
    .select("features", "fare_amount")
    .sample(
        withReplacement=False,
        fraction=0.01,
        seed=42
    )
    .cache()
)

val_small = (
    val_df
    .select("features", "fare_amount")
    .sample(
        withReplacement=False,
        fraction=0.05,
        seed=42
    )
    .cache()
)


train_count = train_small.count()
val_count   = val_small.count()

print(f"Train sample: {train_count:,} rows")
print(f"Validation sample: {val_count:,} rows")


rmse_evaluator = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="rmse"
)

r2_evaluator = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="r2"
)

trained_models = {}
validation_results = []

for model_name, estimator in models_config.items():

    print(f"\nTraining {model_name}...")

    trained_model = estimator.fit(train_small)
    trained_models[model_name] = trained_model

    val_predictions = trained_model.transform(val_small)

    rmse = rmse_evaluator.evaluate(val_predictions)
    r2 = r2_evaluator.evaluate(val_predictions)

    validation_results.append(
        (model_name, __builtins__.round(rmse, 3), __builtins__.round(r2, 4))
    )

    print(f"RMSE: ${rmse:.2f} | R²: {r2:.4f}")

results_df = spark.createDataFrame(
    validation_results,
    ["Model", "RMSE", "R2"]
)

print("\nValidation Results (sorted by RMSE):")
results_df.orderBy("RMSE").show(truncate=False)

train_small.unpersist()
val_small.unpersist()

Train sample: 1,406,796 rows
Validation sample: 683,521 rows

Training 1. Linear Regression...
RMSE: $4.15 | R²: 0.8168

Training 2. Gradient Boosted Trees...
RMSE: $4.15 | R²: 0.8170

Training 3. Random Forest...
RMSE: $4.78 | R²: 0.7573

Training 4. Ridge Regression...
RMSE: $4.43 | R²: 0.7914

Validation Results (sorted by RMSE):
+-------------------------+-----+------+
|Model                    |RMSE |R2    |
+-------------------------+-----+------+
|2. Gradient Boosted Trees|4.152|0.817 |
|1. Linear Regression     |4.154|0.8168|
|4. Ridge Regression      |4.433|0.7914|
|3. Random Forest         |4.781|0.7573|
+-------------------------+-----+------+



DataFrame[features: vector, fare_amount: double]

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

test_data = test_df.select("features", "fare_amount")

rmse_evaluator = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="rmse"
)

mae_evaluator = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="mae"
)

r2_evaluator = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="r2"
)

print(f"{'Model':<25} {'Test RMSE ($)':<15} {'Test MAE ($)':<15} {'Test R²':<10}")

test_results = {}

for model_name, trained_model in trained_models.items():

    print(f"\nEvaluating on test set: {model_name}")

    test_predictions = trained_model.transform(test_data)

    rmse = rmse_evaluator.evaluate(test_predictions)
    mae = mae_evaluator.evaluate(test_predictions)
    r2 = r2_evaluator.evaluate(test_predictions)

    test_results[model_name] = {
        "Test_RMSE": __builtins__.round(rmse, 3),
        "Test_MAE": __builtins__.round(mae, 3),
        "Test_R2": __builtins__.round(r2, 4)
    }

    print(f"  RMSE: ${rmse:.2f} | MAE: ${mae:.2f} | R²: {r2:.4f}")

print("\nFINAL RANKING:")

sorted_results = sorted(
    test_results.items(),
    key=lambda item: item[1]["Test_RMSE"]
)

for rank, (model_name, scores) in enumerate(sorted_results, start=1):
    print(
        f"{rank}. {model_name:<23} "
        f"RMSE=${scores['Test_RMSE']:6.2f}  "
        f"MAE=${scores['Test_MAE']:6.2f}  "
        f"R²={scores['Test_R2']:7.4f}"
    )

best_model_name = sorted_results[0][0]
print(f"\nBEST MODEL: {best_model_name}")

Model                     Test RMSE ($)   Test MAE ($)    Test R²   

Evaluating on test set: 1. Linear Regression
  RMSE: $4.20 | MAE: $2.12 | R²: 0.8101

Evaluating on test set: 2. Gradient Boosted Trees
  RMSE: $4.22 | MAE: $2.04 | R²: 0.8088

Evaluating on test set: 3. Random Forest
  RMSE: $4.83 | MAE: $2.61 | R²: 0.7497

Evaluating on test set: 4. Ridge Regression
  RMSE: $4.48 | MAE: $2.37 | R²: 0.7844

FINAL RANKING:
1. 1. Linear Regression    RMSE=$  4.20  MAE=$  2.12  R²= 0.8101
2. 2. Gradient Boosted Trees RMSE=$  4.22  MAE=$  2.04  R²= 0.8088
3. 4. Ridge Regression     RMSE=$  4.48  MAE=$  2.37  R²= 0.7844
4. 3. Random Forest        RMSE=$  4.83  MAE=$  2.61  R²= 0.7497

BEST MODEL: 1. Linear Regression


In [ ]:
from pyspark.ml.regression import (
    GBTRegressor,
    RandomForestRegressor,
    LinearRegression
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

print(f"\nTUNING BEST MODEL: {best_model_name}")

if "Gradient Boosted Trees" in best_model_name:
    estimator = GBTRegressor(
        featuresCol="features",
        labelCol="fare_amount"
    )

    param_grid = (
        ParamGridBuilder()
        .addGrid(estimator.maxDepth, [3, 5])
        .addGrid(estimator.maxIter, [10, 20])
        .build()
    )

elif "Random Forest" in best_model_name:
    estimator = RandomForestRegressor(
        featuresCol="features",
        labelCol="fare_amount"
    )

    param_grid = (
        ParamGridBuilder()
        .addGrid(estimator.maxDepth, [4, 6])
        .addGrid(estimator.numTrees, [10, 20])
        .build()
    )

else:
    estimator = LinearRegression(
        featuresCol="features",
        labelCol="fare_amount"
    )

    param_grid = (
        ParamGridBuilder()
        .addGrid(estimator.regParam, [0.01, 0.1])
        .addGrid(estimator.maxIter, [50, 100])
        .build()
    )

train_cv_sample = (
    train_df
    .select("features", "fare_amount")
    .sample(
        withReplacement=False,
        fraction=0.01,
        seed=42
    )
    .cache()
)
train_cv_sample.count()

evaluator = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="rmse"
)

cv = CrossValidator(
    estimator=estimator,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=2
)

print("Running 2-fold cross-validation on sampled data...")

cv_model = cv.fit(train_cv_sample)

test_data = test_df.select("features", "fare_amount")
test_predictions = cv_model.bestModel.transform(test_data)

test_rmse = evaluator.evaluate(test_predictions)

print("\nBest Model Parameters:")
print(cv_model.bestModel.extractParamMap())

print(f"\nTuned Test RMSE: ${test_rmse:.2f}")

train_cv_sample.unpersist()


TUNING BEST MODEL: 1. Linear Regression
Running 2-fold cross-validation on sampled data...

Best Model Parameters:
{Param(parent='LinearRegression_6080e0a6655f', name='aggregationDepth', doc='suggested depth for treeAggregate (>= 2).'): 2, Param(parent='LinearRegression_6080e0a6655f', name='elasticNetParam', doc='the ElasticNet mixing parameter, in range [0, 1]. For alpha = 0, the penalty is an L2 penalty. For alpha = 1, it is an L1 penalty.'): 0.0, Param(parent='LinearRegression_6080e0a6655f', name='epsilon', doc='The shape parameter to control the amount of robustness. Must be > 1.0. Only valid when loss is huber'): 1.35, Param(parent='LinearRegression_6080e0a6655f', name='featuresCol', doc='features column name.'): 'features', Param(parent='LinearRegression_6080e0a6655f', name='fitIntercept', doc='whether to fit an intercept term.'): True, Param(parent='LinearRegression_6080e0a6655f', name='labelCol', doc='label column name.'): 'fare_amount', Param(parent='LinearRegression_6080e0a6

DataFrame[features: vector, fare_amount: double]

In [ ]:
best_model = None
best_model_name = None

for model_name, trained_model in trained_models.items():
    if model_name == best_model_name or best_model is None:
        best_model = trained_model
        best_model_name = model_name

feature_names = [
    "passenger_count",
    "trip_distance",
    "haversine_km",
    "trip_duration_min",
    "pickup_hour",
    "is_peak_hour",
    "is_weekend",
    "dow_vector"
]

print(f"\nMODEL INTERPRETABILITY ({best_model_name})")
print(f"{'Feature':<22} {'Value'}")

if hasattr(best_model, "featureImportances"):

    importances = best_model.featureImportances.toArray()

    for feature, importance in zip(feature_names, importances):
        print(f"{feature:<22} {importance:.4f}")

elif hasattr(best_model, "coefficients"):

    coefficients = best_model.coefficients.toArray()

    for feature, coef in zip(feature_names, coefficients):
        print(f"{feature:<22} {coef:+.4f}")

    print(f"\nIntercept: {best_model.intercept:+.4f}")

else:
    print("Model does not support interpretability output.")


MODEL INTERPRETABILITY (1. Linear Regression)
Feature                Value
passenger_count        +0.0000
trip_distance          +7.6020
haversine_km           +0.0000
trip_duration_min      +0.0076
pickup_hour            +0.0577
is_peak_hour           -0.0710
is_weekend             +0.0000
dow_vector             +0.0000

Intercept: +10.6672


In [ ]:
from pyspark.sql import functions as F

print("\nRESIDUAL ANALYSIS")

test_data = test_df.select("features", "fare_amount")

best_model = trained_models[best_model_name]

test_predictions = best_model.transform(test_data)

test_with_residuals = test_predictions.withColumn(
    "residual",
    F.col("fare_amount") - F.col("prediction")
)

test_with_residuals.select(
    "prediction",
    "fare_amount",
    "residual"
).show(10, truncate=False)

residual_stats = test_with_residuals.agg(
    F.round(F.avg("residual"), 2).alias("mean_residual"),
    F.round(F.stddev("residual"), 2).alias("residual_std")
)

print("\nResidual Summary Statistics:")
residual_stats.show()


RESIDUAL ANALYSIS
+------------------+-----------+--------------------+
|prediction        |fare_amount|residual            |
+------------------+-----------+--------------------+
|8.872877973158772 |9.0        |0.12712202684122786 |
|5.976120256675581 |6.0        |0.02387974332441889 |
|23.695509210002886|25.5       |1.8044907899971143  |
|9.100828071832066 |12.5       |3.3991719281679345  |
|6.325052650207503 |8.0        |1.6749473497924967  |
|9.78073686017588  |9.5        |-0.28073686017587995|
|7.324149318844771 |7.5        |0.17585068115522873 |
|6.885139081227383 |6.5        |-0.3851390812273827 |
|9.120238770368669 |11.0       |1.879761229631331   |
|7.783680772880947 |9.5        |1.7163192271190528  |
+------------------+-----------+--------------------+
only showing top 10 rows

Residual Summary Statistics:
+-------------+------------+
|mean_residual|residual_std|
+-------------+------------+
|         1.64|        3.87|
+-------------+------------+



In [ ]:
from pyspark.sql import functions as F

print("\nSAVING MODELS & EXPORTING RESULTS")

safe_model_name = best_model_name.replace(" ", "_").lower()

best_model.write() \
    .overwrite() \
    .save(f"{BASE_PATH}/tests/{safe_model_name}")

if "cv_model" in globals():
    cv_model.bestModel.write() \
        .overwrite() \
        .save(f"{BASE_PATH}/tests/tuned_best_model")

test_data = test_df.select("features", "fare_amount")

test_predictions = best_model.transform(test_data)

test_with_residuals = test_predictions.withColumn(
    "residual",
    F.col("fare_amount") - F.col("prediction")
)

(
    test_with_residuals
        .select("prediction", "fare_amount", "residual")
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header", "true")
        .csv(f"{BASE_PATH}/tableau/final_predictions")
)

print(f"Best model saved at:  {BASE_PATH}/tests/{safe_model_name}")

if "cv_model" in globals():
    print(f"Tuned model saved at: {BASE_PATH}/tests/tuned_best_model")

print(f"Tableau export saved at: {BASE_PATH}/tableau/final_predictions")
''
print("\nBEST MODEL PERFORMANCE:")

if best_model_name in test_results:
    scores = test_results[best_model_name]
    print(
        f"{best_model_name} | "
        f"RMSE = ${scores['Test_RMSE']:.2f}, "
        f"R² = {scores['Test_R2']:.4f}"
    )


SAVING MODELS & EXPORTING RESULTS
Best model saved at:  /content/drive/MyDrive/NYC_Taxi_Yellow_2012/tests/1._linear_regression
Tuned model saved at: /content/drive/MyDrive/NYC_Taxi_Yellow_2012/tests/tuned_best_model
Tableau export saved at: /content/drive/MyDrive/NYC_Taxi_Yellow_2012/tableau/final_predictions

BEST MODEL PERFORMANCE:
1. Linear Regression | RMSE = $4.20, R² = 0.8101


In [ ]:
train_df = spark.read.parquet(
    "/content/drive/MyDrive/NYC_Taxi_Yellow_2012/data/final/train_2012"
)

train_df.show(5)
train_df.printSchema()

+--------------------+-----------+
|            features|fare_amount|
+--------------------+-----------+
|[0.23388725290759...|       18.1|
|[-0.5203900857158...|        7.7|
|[2.49671926877783...|        4.9|
|[-0.5203900857158...|        9.3|
|[0.23388725290759...|        3.3|
+--------------------+-----------+
only showing top 5 rows
root
 |-- features: vector (nullable = true)
 |-- fare_amount: double (nullable = true)



In [ ]:
import pandas as pd
import os
from pyspark.ml.regression import LinearRegressionModel, RandomForestRegressionModel, GBTRegressionModel



BASE_PATH = "/content/drive/MyDrive/NYC_Taxi_Yellow_2012"



if 'trained_models' in locals() and 'best_model_name' in locals():
    best_model = trained_models[best_model_name]
else:
    raise NameError(
        "Required variables 'trained_models' or 'best_model_name' are not defined.\n" +
        "Please ensure preceding cells (especially 'cBwEPgXyDvNo' and 'HoX2YZbYEDs_') have been executed."
    )



importances_array = []
if isinstance(best_model, (RandomForestRegressionModel, GBTRegressionModel)):
    if hasattr(best_model, "featureImportances"):
        importances_array = best_model.featureImportances.toArray()
    else:
        print(f"Warning: Model {type(best_model).__name__} does not have 'featureImportances'.")
elif isinstance(best_model, LinearRegressionModel):
    if hasattr(best_model, "coefficients"):
        importances_array = best_model.coefficients.toArray()
    else:
        print(f"Warning: Model {type(best_model).__name__} does not have 'coefficients'.")
else:
    print(f"Warning: Model type {type(best_model)} is not supported for feature importance/coefficient extraction.")



numeric_features_names = [
    "passenger_count",
    "trip_distance",
    "congestion_surcharge",
    "pickup_hour",
    "is_peak_hour",
    "is_weekend"
]
one_hot_encoded_dow_names = [f"dow_vector_day_{i}" for i in range(7)]

feature_names = numeric_features_names + one_hot_encoded_dow_names



if len(feature_names) != len(importances_array):
    print(f"Warning: Mismatch between reconstructed feature names count ({len(feature_names)}) and actual importances count ({len(importances_array)}). Using generic names as fallback.")

    feature_names = [f"feature_{i}" for i in range(len(importances_array))]


fi_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances_array
})

fi_df = fi_df.sort_values("Importance", ascending=False)


os.makedirs(f"{BASE_PATH}/tableau", exist_ok=True)
fi_df.to_csv(f"{BASE_PATH}/tableau/feature_importance.csv", index=False)

print(f"Feature importance data saved to: {BASE_PATH}/tableau/feature_importance.csv")


Feature importance data saved to: /content/drive/MyDrive/NYC_Taxi_Yellow_2012/tableau/feature_importance.csv


In [ ]:

import pandas as pd
import os
from pyspark.ml.regression import (
    LinearRegressionModel,
    GBTRegressionModel,
    RandomForestRegressionModel
)
from pyspark.sql import SparkSession
from pyspark.ml.evaluation import RegressionEvaluator
import re

spark = SparkSession.builder.getOrCreate()


BASE_PATH = "/content/drive/MyDrive/NYC_Taxi_Yellow_2012"


if 'best_model_name' not in locals():
    print("Warning: best_model_name not found in current scope. Defaulting to '2. Gradient Boosted Trees'.")
    best_model_name = "2. Gradient Boosted Trees"


if 'trained_models' in globals() and best_model_name in trained_models:
    best_model = trained_models[best_model_name]
else:
    print(f"Warning: 'trained_models' or '{best_model_name}' not found in current scope. Attempting to load best model from disk.")

    safe_model_name = best_model_name.replace(" ", "_").lower()
    model_path = f"{BASE_PATH}/tests/{safe_model_name}"

    try:

        if "linear regression" in safe_model_name:
            best_model = LinearRegressionModel.load(model_path)
        elif "gradient boosted trees" in safe_model_name:
            best_model = GBTRegressionModel.load(model_path)
        elif "random forest" in safe_model_name:
            best_model = RandomForestRegressionModel.load(model_path)
        else:
            raise ValueError(f"Unknown model type for '{best_model_name}'. Cannot load from disk.")

        print(f"Successfully loaded '{best_model_name}' from {model_path}")
    except Exception as e:
        raise RuntimeError(f"Could not load best_model from disk at {model_path}. Ensure previous cells were executed to train and save the model, or check the path. Error: {e}")


test_df = spark.read.parquet(f"{BASE_PATH}/data/final/test_2012")

evaluator_rmse = RegressionEvaluator(labelCol="fare_amount", metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol="fare_amount", metricName="mae")
evaluator_r2 = RegressionEvaluator(labelCol="fare_amount", metricName="r2")

predictions = best_model.transform(test_df)

rmse = evaluator_rmse.evaluate(predictions)
mae = evaluator_mae.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

metrics_df = pd.DataFrame({
    "Model": [re.sub(r"^\d+\.\s*", "", best_model_name).strip()],
    "RMSE": [rmse],
    "MAE": [mae],
    "R2": [r2]
})

metrics_df.to_csv(f"{BASE_PATH}/tableau/model_comparison.csv", index=False)